In [13]:
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.metrics import matthews_corrcoef, precision_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split, StratifiedKFold, TimeSeriesSplit
from xgboost import XGBClassifier
import pickle
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
from hmmlearn.hmm import GaussianHMM
import re
import itertools
import time
import warnings
warnings.filterwarnings("ignore", module="joblib")
import databento as db

# Read the DBN file into a DBNStore object
dbn_store = db.DBNStore.from_file('qqq_1m.dbn')
# Convert the data to a pandas DataFrame for analysis
df = dbn_store.to_df()
df_main = df.reset_index()[['symbol', 'ts_event', 'close', 'open', 'high', 'low', 'volume']].copy()

df_main['datetime_est'] = (df_main['ts_event'].dt.tz_convert('America/New_York'))
df_first = df_main[(df_main['datetime_est'].dt.hour == 9) & (df_main['datetime_est'].dt.minute == 0)]
df_last = df_main[(df_main['datetime_est'].dt.hour == 15) & (df_main['datetime_est'].dt.minute == 59)]

# 1. Session Structure & Market Phases

In [64]:
def add_intraday_labels(df: pd.DataFrame, dt_col: str = "datetime_est") -> pd.DataFrame:
    
    out = df.copy()

    # Ensure datetime
    out[dt_col] = pd.to_datetime(out[dt_col], errors="coerce")
    if out[dt_col].isna().any():
        bad = out[dt_col].isna().sum()
        raise ValueError(f"{bad} rows in {dt_col} could not be parsed to datetime.")

    # Extract time-of-day in minutes since midnight (ET)
    tod_minutes = out[dt_col].dt.hour * 60 + out[dt_col].dt.minute
    out["_tod_minutes"] = tod_minutes

    # Open time (09:30 ET) in minutes
    premarket_min = 7 * 60  # 420
    open_min = 9 * 60 + 30  # 570
    close_min = 16 * 60 - 1   # 960

    # Minutes since open (can be negative pre-market, positive post-open)
    out["minutes_since_open"] = out["_tod_minutes"] - open_min

    # Column 1: simple session label
    out["session_simple"] = np.select(
        [
            out["_tod_minutes"] < premarket_min,
            (out["_tod_minutes"] >= premarket_min) & (out["_tod_minutes"] < open_min),
            (out["_tod_minutes"] >= open_min) & (out["_tod_minutes"] <= close_min),
            out["_tod_minutes"] > close_min,
        ],
        ["overnight", "pre_market", "open_market", "post_market"],
        default=np.nan
    )

    # Column 2: detailed session label (your buckets)
    out["session_detail"] = np.select(
        [
            # Pre-market buckets
            (out["_tod_minutes"] < 7 *60),
            (out["_tod_minutes"] >= 7*60) & (out["_tod_minutes"] < 9*60),
            (out["_tod_minutes"] >= 9*60) & (out["_tod_minutes"] < open_min),

            # Open market buckets
            (out["_tod_minutes"] >= open_min) & (out["_tod_minutes"] < 9*60+45),
            (out["_tod_minutes"] >= 9*60+45) & (out["_tod_minutes"] < 10*60),
            (out["_tod_minutes"] >= 10*60) & (out["_tod_minutes"] < 12*60),
            (out["_tod_minutes"] >= 12*60) & (out["_tod_minutes"] < 14*60),
            (out["_tod_minutes"] >= 14*60) & (out["_tod_minutes"] < 15*60+30),
            (out["_tod_minutes"] >= 15*60+30) & (out["_tod_minutes"] < 15*60+45),
            (out["_tod_minutes"] >= 15*60+45) & (out["_tod_minutes"] <= close_min),

            # Post-market buckets
            (out["_tod_minutes"] > close_min) & (out["_tod_minutes"] < 16*60+15),
            (out["_tod_minutes"] >= 16*60+15) & (out["_tod_minutes"] < 17*60),
            (out["_tod_minutes"] >= 17*60) & (out["_tod_minutes"] <= 20*60),
        ],
        [
            "overnight",
            "early_pre_market",
            "late_pre_market",
            "early_open",
            "late_open",
            "morning",
            "midday",
            "late_day",
            "early_close",
            "late_close",
            "early_post_market",
            "late_post_market",
            "post_market",
        ],
        default="other"
    )

    # Cleanup
    out = out.drop(columns=["_tod_minutes", "_detail_simple_check"], errors="ignore")
    return out

#df_intraday_labels[df_intraday_labels['minutes_since_open'] == 390]
#Shortest minutes_since_open = -330 largest is 629. 0 = 930am, 389 = 4:00pm
df_intraday_labels = add_intraday_labels(df_main)
df_intraday_labels['Date'] = pd.to_datetime(df_intraday_labels['datetime_est']).dt.strftime('%Y-%m-%d')

In [56]:
ticker = 'QQQ'
df_yf = yf.Ticker(ticker).history(start='2018-05-01', end='2025-12-20', interval="1d", auto_adjust=True).reset_index()[['Date', 'Close', 'High', 'Low', 'Volume']]
df_yf['Date'] = pd.to_datetime(df_yf['Date']).dt.strftime('%Y-%m-%d')

cols = ['Date', 'open']
db_close = df_intraday_labels[cols][df_intraday_labels['minutes_since_open'] == 390].copy()
db_open = df_intraday_labels[cols][df_intraday_labels['minutes_since_open'] == 0].copy()